In [ ]:
#Install required libraries
Bash
pip install numpy matplotlib scikit-learn torch torchvision gpytorch seaborn

#Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, accuracy_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
from scipy.stats import entropy
import gpytorch

#Regression Experiment (Noisy Sine Wave)
Paper specification:
•	y=sin(x)+ϵy = sin(x) + ϵ
•	ε ~ N(0, 0.1)
•	200 training points
•	100 test points

Generate Dataset
np.random.seed(42)
n_train = 200
n_test = 100

X = np.linspace(-4, 4, n_train + n_test)
y_true = np.sin(X)

noise = np.random.normal(0, 0.1, size=len(X))
y = y_true + noise

X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

#Deterministic Neural Network
class DeterministicNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1,64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self,x):
        return self.net(x)

model = DeterministicNN()
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(1000):
    optimizer.zero_grad()
    loss = loss_fn(model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

pred_det = model(X_test_t).detach().numpy()

#Monte Carlo Dropout
Dropout kept active during inference.

class MCDropoutNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1,64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,1)
        )

    def forward(self,x):
        return self.net(x)

mc_model = MCDropoutNN()
optimizer = optim.Adam(mc_model.parameters(), lr=0.01)

for epoch in range(1000):
    optimizer.zero_grad()
    loss = loss_fn(mc_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

Monte Carlo Sampling (50 samples)
mc_model.train()   # keep dropout ON
samples = []
for _ in range(50):
    samples.append(mc_model(X_test_t).detach().numpy())

samples = np.array(samples)

mc_mean = samples.mean(axis=0)
mc_std = samples.std(axis=0)

#Bayesian Neural Network (Variational Approximation)
Simplified VI implementation.
class BayesianLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight_mu = nn.Parameter(torch.zeros(out_features, in_features))
        self.weight_logvar = nn.Parameter(torch.zeros(out_features, in_features))

    def forward(self, x):
        eps = torch.randn_like(self.weight_mu)
        weight = self.weight_mu + torch.exp(0.5*self.weight_logvar)*eps
        return torch.matmul(x, weight.t())


class BayesianNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.b1 = BayesianLayer(1,64)
        self.b2 = BayesianLayer(64,1)

    def forward(self,x):
        x = torch.relu(self.b1(x))
        return self.b2(x)

bnn = BayesianNN()
optimizer = optim.Adam(bnn.parameters(), lr=0.01)

for epoch in range(1000):
    optimizer.zero_grad()
    loss = loss_fn(bnn(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

Posterior Sampling
bnn_samples = []
for _ in range(50):
    bnn_samples.append(bnn(X_test_t).detach().numpy())
bnn_samples = np.array(bnn_samples)
bnn_mean = bnn_samples.mean(axis=0)
bnn_std = bnn_samples.std(axis=0)

#Gaussian Process Regression
class GPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.RBFKernel()

    def forward(self, x):
        mean = self.mean_module(x)
        covar = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean, covar)

likelihood = gpytorch.likelihoods.GaussianLikelihood()
gp_model = GPModel(X_train_t.squeeze(), y_train_t.squeeze(), likelihood)

gp_model.train()
likelihood.train()

optimizer = torch.optim.Adam(gp_model.parameters(), lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, gp_model)

for i in range(200):
    optimizer.zero_grad()
    output = gp_model(X_train_t.squeeze())
    loss = -mll(output, y_train_t.squeeze())
    loss.backward()
    optimizer.step()

GP Prediction
gp_model.eval()
likelihood.eval()
with torch.no_grad():
    pred = likelihood(gp_model(X_test_t.squeeze()))
    gp_mean = pred.mean.numpy()
    gp_std = pred.stddev.numpy()

#Regression Uncertainty Plot
plt.figure(figsize=(8,5))

plt.scatter(X_train, y_train, color='blue', s=10)
plt.plot(X_test, np.sin(X_test), 'r', label='True')

plt.plot(X_test, mc_mean, label="MC Dropout")
plt.fill_between(
    X_test,
    mc_mean.flatten()-2*mc_std.flatten(),
    mc_mean.flatten()+2*mc_std.flatten(),
    alpha=0.3
)
plt.legend()
plt.title("Bayesian Regression with Uncertainty")
plt.show()

#Classification Dataset (Two Gaussian Clusters)
from sklearn.datasets import make_blobs

X, y = make_blobs(
    n_samples=600,
    centers=2,
    cluster_std=2.5,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.3
)

X_train_t = torch.tensor(X_train,dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)

X_test_t = torch.tensor(X_test,dtype=torch.float32)

#Classification Neural Network
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2,64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.net(x)

clf = Classifier()
optimizer = optim.Adam(clf.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

for epoch in range(500):
    optimizer.zero_grad()
    loss = loss_fn(clf(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

#Monte Carlo Dropout Classification
clf.train()
probs = [ ]
for _ in range(50):
    probs.append(clf(X_test_t).detach().numpy())
probs = np.array(probs)
mean_prob = probs.mean(axis=0)

#Evaluation Metrics
Accuracy
acc = accuracy_score(y_test, mean_prob>0.5)
Brier Score
brier = brier_score_loss(y_test, mean_prob)
Negative Log-Likelihood
eps = 1e-10
nll = -np.mean(
    y_test*np.log(mean_prob+eps)
    +(1-y_test)*np.log(1-mean_prob+eps)
)

Expected Calibration Error (ECE)
def compute_ece(y_true, prob, bins=10):
    bin_edges = np.linspace(0,1,bins+1)
    ece = 0
    for i in range(bins):
        mask = (prob>=bin_edges[i])&(prob<bin_edges[i+1])
        if np.sum(mask)>0:
            acc = np.mean(y_true[mask])
            conf = np.mean(prob[mask])
            ece += np.abs(acc-conf)*np.sum(mask)/len(prob)
    return ece
ece = compute_ece(y_test, mean_prob.flatten())

#Calibration Plot (Figure 3)
prob_true, prob_pred = calibration_curve(
    y_test,
    mean_prob.flatten(),
    n_bins=10
)

plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.title("Calibration Curve")
plt.show()


SyntaxError: invalid character '•' (U+2022) (1362677682.py, line 23)